# 03 — HeteroGAT (Epic 3)

**Two tasks (multi-task head):**
- **Node-level**: predict per-node criticality (`data['compute'].y`)  → Spearman ρ
- **Graph-level**: predict total step time (`data.y`) → MSE + ΔT%

**Two model variants:**
- **Bidirectional** — standard GAT, messages flow both directions (associational baseline)
- **Directed** — messages flow ancestor → descendant only (causal inductive bias)

**Acceptance criteria (Sprint 5):**
- Directed variant: Spearman ρ > 0.70 on val set
- Both variants: ΔT% < 5% on val set
- Both val loss curves decreasing

**Data:** `archi829/bottleneck-oracle-graphs` on HuggingFace (501 graphs)

**No MLP/baseline comparison in this notebook** — baselines are in `model_1f1b.ipynb`.


In [ ]:
!pip install -q torch-geometric huggingface_hub scikit-learn scipy

In [ ]:
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
from huggingface_hub import list_repo_files, hf_hub_download
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

os.makedirs('data', exist_ok=True)


## 1. Load graphs from HuggingFace

In [ ]:
REPO = "archi829/bottleneck-oracle-graphs"

files = [
    f for f in list_repo_files(REPO, repo_type="dataset")
    if f.endswith(".pt") and not f.startswith("data/")
]
print(f"Found {len(files)} graph files")

graphs = []
for fname in sorted(files):
    path = hf_hub_download(repo_id=REPO, filename=fname, repo_type="dataset")
    graphs.append(torch.load(path, weights_only=False))

print(f"Loaded {len(graphs)} graphs")

# ── quick sanity on one graph ─────────────────────────────────────────────────
g0 = graphs[0]
assert g0['compute'].x.shape[1] == 4, "Expected 4 compute features"
assert g0['compute'].y is not None,    "Missing node labels"
assert g0.y is not None,               "Missing graph label"
print(f"Graph schema: {g0}")
print(f"Compute nodes: {g0['compute'].x.shape[0]}")
print(f"Edge types: {g0.edge_types}")


## 2. Train / Val / Test split

**Same SEED and 70/15/15 ratios as `model_1f1b.ipynb`** so the held-out test set is identical
across all notebooks.  If `data/splits.pt` exists (written by `model_1f1b`) we load it;
otherwise we recompute with the same seed.


In [ ]:
SPLITS_PATH = 'data/splits.pt'

if os.path.exists(SPLITS_PATH):
    splits   = torch.load(SPLITS_PATH)
    train_idx = splits['train']
    val_idx   = splits['val']
    test_idx  = splits['test']
    print('Loaded splits from data/splits.pt')
else:
    n = len(graphs)
    idx = list(range(n))
    train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=SEED)
    val_idx,   test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED)
    torch.save({'train': train_idx, 'val': val_idx, 'test': test_idx}, SPLITS_PATH)
    print('Splits recomputed and saved.')

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[i] for i in val_idx]
test_graphs  = [graphs[i] for i in test_idx]
print(f"Split → train:{len(train_graphs)}  val:{len(val_graphs)}  test:{len(test_graphs)}")


## 3. HeteroGAT model

### Node ordering contract (IMPORTANT — read before editing)

Graphs from `02-trace-generation-fixed.ipynb` assign node IDs by timestamp order
(`ops = sorted(ops, key=lambda x: x['ts'])`), and all edges are `i-1 → i`.
`compute_idx` is built from `G.nodes()` iteration, which preserves insertion
(= timestamp) order.

**Consequence:** for every `('compute','depends_on','compute')` edge `(src, dst)`,
`src < dst` is guaranteed by construction.

The directed variant exploits this by masking `edge_index` to keep only `src < dst`.
If `02` ever changes to add non-sequential edges or reindex nodes differently,
this assumption will silently break.  The assertion in `_safe_directed_mask` will
catch that at runtime.

### Architecture

- 2-layer HeteroConv with GATConv kernels
- Edge types used: `compute→compute` (depends_on), `compute→network` (sends_to),
  `network→compute` (feeds)
- Node head: per-compute-node criticality logit → BCE loss
- Graph head: mean-pool compute embeddings → step-time regression → MSE loss


In [ ]:
class HeteroGAT(nn.Module):
    """
    Two-layer heterogeneous GAT.

    Outputs:
      node_logits : [N_compute]  — criticality score (pre-sigmoid)
      step_time   : scalar       — predicted total step time (ms)

    directed=True  → only forward edges (src < dst) passed to conv layers
    directed=False → standard bidirectional GAT
    """
    def __init__(self, in_dim: int = 4, hidden: int = 64,
                 heads: int = 4, directed: bool = False):
        super().__init__()
        self.directed = directed

        # ── Layer 1 ───────────────────────────────────────────────────────────
        self.conv1 = HeteroConv({
            ('compute', 'depends_on', 'compute'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
            ('compute', 'sends_to', 'network'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
            ('network', 'feeds', 'compute'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
        }, aggr='sum')

        # ── Layer 2 ───────────────────────────────────────────────────────────
        self.conv2 = HeteroConv({
            ('compute', 'depends_on', 'compute'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
            ('compute', 'sends_to', 'network'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
            ('network', 'feeds', 'compute'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
        }, aggr='sum')

        # ── heads ─────────────────────────────────────────────────────────────
        self.node_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )
        self.graph_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    # ── directed masking ──────────────────────────────────────────────────────
    @staticmethod
    def _safe_directed_mask(edge_index: torch.Tensor) -> torch.Tensor:
        """
        Keep only forward edges (src < dst).

        Relies on the src<dst guarantee from 02-trace-generation-fixed.ipynb.
        Asserts that at least *some* edges satisfy src<dst (catches reindexing
        regressions immediately rather than silently training on empty graphs).
        """
        if edge_index is None or edge_index.shape[1] == 0:
            return edge_index
        mask = edge_index[0] < edge_index[1]
        assert mask.any(), (
            "Directed masking produced 0 forward edges.  "
            "Check whether 02 still generates timestamp-ordered node IDs. "
            "If graph construction changed, update the masking strategy."
        )
        return edge_index[:, mask]

    def _apply_directed(self, edge_index_dict):
        out = {}
        for k, ei in edge_index_dict.items():
            if ei is not None and ei.shape[1] > 0:
                out[k] = self._safe_directed_mask(ei)
            else:
                out[k] = ei
        return out

    def forward(self, x_dict, edge_index_dict):
        if self.directed:
            edge_index_dict = self._apply_directed(edge_index_dict)

        # ── layer 1 ───────────────────────────────────────────────────────────
        # Only pass edge types that actually exist in this graph
        present = {k: v for k, v in edge_index_dict.items()
                   if v is not None and v.shape[1] > 0}
        h = self.conv1(x_dict, present)
        h = {k: F.elu(v) for k, v in h.items()}
        # carry forward node types not updated by conv (missing in some graphs)
        for k in x_dict:
            if k not in h:
                h[k] = x_dict[k]

        # ── layer 2 ───────────────────────────────────────────────────────────
        present2 = {k: v for k, v in edge_index_dict.items()
                    if v is not None and v.shape[1] > 0}
        h2 = self.conv2(h, present2)
        h2 = {k: F.elu(v) for k, v in h2.items()}
        for k in h:
            if k not in h2:
                h2[k] = h[k]

        compute_emb = h2['compute']                              # [N_c, hidden]
        node_logits = self.node_head(compute_emb).squeeze(-1)   # [N_c]
        graph_emb   = compute_emb.mean(dim=0, keepdim=True)     # [1, hidden]
        step_time   = self.graph_head(graph_emb).squeeze()      # scalar

        return node_logits, step_time


## 4. Training utilities

In [ ]:
def delta_t_pct(pred, true):
    """Absolute % error in step-time prediction."""
    return abs(pred - true) / (true + 1e-8) * 100


def train_one_epoch(model, graphs, optimizer, node_weight=0.5, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    for g in graphs:
        g = g.to(DEVICE)
        optimizer.zero_grad()

        node_logits, step_pred = model(g.x_dict, g.edge_index_dict)

        # graph-level: MSE on step time
        graph_loss = F.mse_loss(step_pred, g.y.squeeze().to(DEVICE))

        # node-level: BCE on criticality labels
        node_labels = g['compute'].y.to(DEVICE)
        node_loss   = F.binary_cross_entropy_with_logits(node_logits, node_labels)

        loss = (1 - node_weight) * graph_loss + node_weight * node_loss
        loss.backward()

        # gradient clipping — prevents rare exploding-gradient spikes in GAT
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(graphs)


@torch.no_grad()
def evaluate(model, graphs):
    model.eval()
    pred_times, true_times = [], []
    node_rho_list = []

    for g in graphs:
        g = g.to(DEVICE)
        node_logits, step_pred = model(g.x_dict, g.edge_index_dict)

        pred_times.append(step_pred.item())
        true_times.append(g.y.item())

        scores = torch.sigmoid(node_logits).cpu().numpy()
        labels = g['compute'].y.cpu().numpy()
        # skip degenerate graphs where all labels are identical
        if labels.std() > 0:
            rho, _ = spearmanr(scores, labels)
            node_rho_list.append(rho)

    mse    = mean_squared_error(true_times, pred_times)
    deltas = [delta_t_pct(p, t) for p, t in zip(pred_times, true_times)]
    spear  = float(np.mean(node_rho_list)) if node_rho_list else float('nan')

    return {
        'mse':       mse,
        'mean_dt':   float(np.mean(deltas)),
        'median_dt': float(np.median(deltas)),
        'spearman':  spear,
    }


## 5. Train — Bidirectional variant

Checkpoint saved to `data/heterogat_bidir_best.pt` on val-MSE improvement.


In [ ]:
EPOCHS  = 80
LR      = 3e-4
NODE_W  = 0.5
CKPT_BIDIR = 'data/heterogat_bidir_best.pt'

model_bidir = HeteroGAT(in_dim=4, hidden=64, heads=4, directed=False).to(DEVICE)
opt_bidir   = torch.optim.Adam(model_bidir.parameters(), lr=LR)

best_val_mse_bidir = float('inf')
print(f"{'Epoch':>6}  {'Train L':>9}  {'Val MSE':>9}  {'Val ΔT%':>8}  {'Val ρ':>7}  {'Saved':>5}")
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model_bidir, train_graphs, opt_bidir, node_weight=NODE_W)

    if epoch % 10 == 0:
        m = evaluate(model_bidir, val_graphs)
        saved = ''
        if m['mse'] < best_val_mse_bidir:
            best_val_mse_bidir = m['mse']
            torch.save(model_bidir.state_dict(), CKPT_BIDIR)
            saved = '✅'
        print(f"{epoch:>6}  {train_loss:>9.4f}  {m['mse']:>9.2f}  "
              f"{m['mean_dt']:>7.2f}%  {m['spearman']:>7.4f}  {saved}")

print(f'\n✅ Bidir done. Best val MSE: {best_val_mse_bidir:.4f}')


## 6. Train — Directed variant

Messages flow ancestor → descendant only (causal inductive bias).
See §3 for the `src < dst` ordering contract.


In [ ]:
CKPT_DIR = 'data/heterogat_dir_best.pt'

model_dir = HeteroGAT(in_dim=4, hidden=64, heads=4, directed=True).to(DEVICE)
opt_dir   = torch.optim.Adam(model_dir.parameters(), lr=LR)

best_val_mse_dir = float('inf')
print(f"{'Epoch':>6}  {'Train L':>9}  {'Val MSE':>9}  {'Val ΔT%':>8}  {'Val ρ':>7}  {'Saved':>5}")
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model_dir, train_graphs, opt_dir, node_weight=NODE_W)

    if epoch % 10 == 0:
        m = evaluate(model_dir, val_graphs)
        saved = ''
        if m['mse'] < best_val_mse_dir:
            best_val_mse_dir = m['mse']
            torch.save(model_dir.state_dict(), CKPT_DIR)
            saved = '✅'
        print(f"{epoch:>6}  {train_loss:>9.4f}  {m['mse']:>9.2f}  "
              f"{m['mean_dt']:>7.2f}%  {m['spearman']:>7.4f}  {saved}")

print(f'\n✅ Directed done. Best val MSE: {best_val_mse_dir:.4f}')


## 7. Final evaluation — held-out test set

Load best checkpoints, run on test set.  Also compute 1F1B analytic baseline
**on the same test split** for reference.

**1F1B formula:**  step time ≈ sum of all compute-node durations (column 0 of x).
This is the sum of raw op durations, i.e. the serial lower bound.


In [ ]:
# ── load best checkpoints ────────────────────────────────────────────────────
model_bidir.load_state_dict(torch.load(CKPT_BIDIR, map_location=DEVICE))
model_dir.load_state_dict(torch.load(CKPT_DIR,   map_location=DEVICE))

bidir_m = evaluate(model_bidir, test_graphs)
dir_m   = evaluate(model_dir,   test_graphs)

# ── 1F1B analytic on same test set ───────────────────────────────────────────
# Formula: predicted step time = sum of all compute-node raw durations.
# (x[:, 0] = duration_ms; we do NOT multiply by y because y is the label we're
#  trying to predict — that would be leaking the answer.)
def predict_1f1b(g):
    return g['compute'].x[:, 0].sum().item()

true_times  = [g.y.item() for g in test_graphs]
f1b_preds   = [predict_1f1b(g) for g in test_graphs]
f1b_mse     = mean_squared_error(true_times, f1b_preds)
f1b_deltas  = [delta_t_pct(p, t) for p, t in zip(f1b_preds, true_times)]

# ── comparison table ─────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"{'Model':<25} {'MSE':>10} {'Mean ΔT%':>10} {'Median ΔT%':>12} {'Spearman ρ':>12}")
print(f"{'-'*72}")
print(f"{'1F1B Analytic (ref)':<25} {f1b_mse:>10.4f} "      f"{np.mean(f1b_deltas):>10.2f} {np.median(f1b_deltas):>12.2f} {'—':>12}")
print(f"{'HeteroGAT (bidir)':<25} {bidir_m['mse']:>10.4f} "      f"{bidir_m['mean_dt']:>10.2f} {bidir_m['median_dt']:>12.2f} "      f"{bidir_m['spearman']:>12.4f}")
print(f"{'HeteroGAT (directed)':<25} {dir_m['mse']:>10.4f} "      f"{dir_m['mean_dt']:>10.2f} {dir_m['median_dt']:>12.2f} "      f"{dir_m['spearman']:>12.4f}")
print(f"{'='*72}")

# ── acceptance criteria check (Sprint 5) ─────────────────────────────────────
print("\n📋 Acceptance criteria:")
rho_ok = dir_m['spearman'] > 0.70
dt_bidir_ok = bidir_m['mean_dt'] < 5.0
dt_dir_ok   = dir_m['mean_dt']   < 5.0
print(f"  Directed Spearman ρ > 0.70 : {dir_m['spearman']:.4f}  {'✅' if rho_ok else '❌'}")
print(f"  Bidir ΔT% < 5%%            : {bidir_m['mean_dt']:.2f}%%  {'✅' if dt_bidir_ok else '❌'}")
print(f"  Directed ΔT% < 5%%         : {dir_m['mean_dt']:.2f}%%  {'✅' if dt_dir_ok else '❌'}")


## 8. Bottleneck inspection — top-5 critical nodes

For one test graph, show which nodes the **directed model** scores highest.
This is the output the Sprint 6 GraphRAG loop will consume.


In [ ]:
model_dir.eval()
g = test_graphs[0].to(DEVICE)

with torch.no_grad():
    node_logits, step_pred = model_dir(g.x_dict, g.edge_index_dict)

scores = torch.sigmoid(node_logits).cpu().numpy()
labels = g['compute'].y.cpu().numpy()
durs   = g['compute'].x[:, 0].cpu().numpy()    # raw duration_ms

top5 = np.argsort(scores)[::-1][:5]
rho_val, _ = spearmanr(scores, labels)

print("Top-5 predicted bottleneck nodes (directed model):")
print(f"{'Node':>6}  {'Score':>7}  {'Label':>7}  {'Dur (ms)':>10}")
print('-' * 38)
for i in top5:
    print(f"{i:>6}  {scores[i]:>7.4f}  {int(labels[i]):>7}  {durs[i]:>10.3f}")

print(f"\nSpearman ρ on this graph : {rho_val:.4f}")
print(f"Predicted step time      : {step_pred.item():.2f} ms")
print(f"True step time           : {g.y.item():.2f} ms")
print(f"ΔT%                      : {delta_t_pct(step_pred.item(), g.y.item()):.2f}%")


## 9. Save checkpoints to HuggingFace

Uncomment and run once training is complete so weights are persisted across sessions.
Both checkpoints go to the same HF dataset repo as the graphs.


In [ ]:
# from huggingface_hub import HfApi, login
# login(token="YOUR_HF_TOKEN_HERE")   # or use Kaggle secrets / Colab secrets
# api = HfApi()
#
# for ckpt in [CKPT_BIDIR, CKPT_DIR]:
#     api.upload_file(
#         path_or_fileobj=ckpt,
#         path_in_repo=f"checkpoints/{os.path.basename(ckpt)}",
#         repo_id=REPO,
#         repo_type='dataset',
#     )
#     print(f'✅ Uploaded {ckpt}')
